# Jigsaw Unintended Bias — 3rd Place Solution
Faithful reimplementation of the group's methods based on: [sakami0000/kaggle_jigsaw](https://github.com/sakami0000/kaggle_jigsaw).

**Procedures:**
1. Setup & data download  
2. Preprocessing & sample weighting  
3. BERT-large-uncased fine-tuning (head+tail truncation, negative downsampling, multi-task loss)  
4. LSTM-GRU baseline (GloVe + FastText embeddings, Beta-smoothed annotator-count loss)  
5. Optuna ensemble blending  
6. Evaluation & Conclusion


## 1. Environment Setup

In [ ]:
# Install dependencies (Colab already has torch/transformers; install legacy bert library + optuna)
!pip install -q pytorch-pretrained-bert==0.6.2 optuna==3.6.1 kaggle


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.7/86.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.1/380.1 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 169.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.2 MB/s eta 0:00:00


In [ ]:
import os, json, random, time, warnings
from pathlib import Path
from collections import Counter
from itertools import chain

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

import torch
from torch import nn
import torch.nn.functional as F

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 2. Download Data


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Paths ────────────────────────────────────────────────────────────────────
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/jigsaw-data')
DATA_DIR       = Path('/content/data')
DATA_DIR.mkdir(exist_ok=True)
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH  = DATA_DIR / 'test.csv'

if (DRIVE_DATA_DIR / 'train.csv').exists():
    # ── Fast path: copy from Drive (no download) ────────────────────────────
    print('Found data on Google Drive. Copying to local runtime...')
    !cp "{DRIVE_DATA_DIR}/train.csv" "{TRAIN_PATH}"
    !cp "{DRIVE_DATA_DIR}/test.csv"  "{TEST_PATH}"
else:
    # ── Slow path: download from Kaggle, then cache to Drive ────────────────
    from google.colab import files
    if not os.path.exists('/content/kaggle.json'):
        print("kaggle.json not found. Please upload it.")
        files.upload()

    !mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle competitions download -c jigsaw-unintended-bias-in-toxicity-classification -p /content/data
    !unzip -o /content/data/jigsaw-unintended-bias-in-toxicity-classification.zip -d /content/data

    # Persist to Drive for next time
    DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
    !cp "{TRAIN_PATH}" "{DRIVE_DATA_DIR}/"
    !cp "{TEST_PATH}"  "{DRIVE_DATA_DIR}/"
    print('Data cached to Google Drive.')

assert TRAIN_PATH.exists(), f'Missing: {TRAIN_PATH}'
assert TEST_PATH.exists(),  f'Missing: {TEST_PATH}'
print('Data ready.')

Mounted at /content/drive
Found data on Google Drive. Copying to local runtime...
Data ready.


## 3. Constants & Column Definitions

### Inspecting DataFrames

In [ ]:
# Load data for inspection
print("Loading data for inspection...")
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

print(f"Shape of training data: {df_train.shape}")
print(df_train.columns)
print(f"Shape of test data: {df_test.shape}")

print("\nFirst 5 rows of training data:")
display(df_train.head())

print("\nFirst 5 rows of test data:")
display(df_test.head())

Loading data for inspection...
Shape of training data: (1804874, 45)
Index(['id', 'target', 'comment_text', 'severe_toxicity', 'obscene',
       'identity_attack', 'insult', 'threat', 'asian', 'atheist', 'bisexual',
       'black', 'buddhist', 'christian', 'female', 'heterosexual', 'hindu',
       'homosexual_gay_or_lesbian', 'intellectual_or_learning_disability',
       'jewish', 'latino', 'male', 'muslim', 'other_disability',
       'other_gender', 'other_race_or_ethnicity', 'other_religion',
       'other_sexual_orientation', 'physical_disability',
       'psychiatric_or_mental_illness', 'transgender', 'white', 'created_date',
       'publication_id', 'parent_id', 'article_id', 'rating', 'funny', 'wow',
       'sad', 'likes', 'disagree', 'sexual_explicit',
       'identity_annotator_count', 'toxicity_annotator_count'],
      dtype='object')
Shape of test data: (97320, 2)

First 5 rows of training data:


,id,target,comment_text,severe_toxicity,obscene,identity_attack,insult,threat,asian,atheist,...,article_id,rating,funny,wow,sad,likes,disagree,sexual_explicit,identity_annotator_count,toxicity_annotator_count
0,59848,0.000000,"This is so cool. It's like, 'would you want yo...",0.000000,0.0,0.000000,0.00000,0.0,NaN,NaN,...,2006,rejected,0,0,0,0,0,0.0,0,4
1,59849,0.000000,Thank you!! This would make my life a lot less...,0.000000,0.0,0.000000,0.00000,0.0,NaN,NaN,...,2006,rejected,0,0,0,0,0,0.0,0,4
2,59852,0.000000,This is such an urgent design problem; kudos t...,0.000000,0.0,0.000000,0.00000,0.0,NaN,NaN,...,2006,rejected,0,0,0,0,0,0.0,0,4
3,59855,0.000000,Is this something I'll be able to install on m...,0.000000,0.0,0.000000,0.00000,0.0,NaN,NaN,...,2006,rejected,0,0,0,0,0,0.0,0,4
4,59856,0.893617,haha you guys are a bunch of losers.,0.021277,0.0,0.021277,0.87234,0.0,0.0,0.0,...,2006,rejected,0,0,0,1,0,0.0,4,47



First 5 rows of test data:


,id,comment_text
0,7097320,[ Integrity means that you pay your debts.]\n\...
1,7097321,This is malfeasance by the Administrator and t...
2,7097322,@Rmiller101 - Spoken like a true elitist. But ...
3,7097323,"Paul: Thank you for your kind words. I do, in..."
4,7097324,Sorry you missed high school. Eisenhower sent ...


In [ ]:
TOXICITY_COLUMN    = 'target'
IDENTITY_COLUMNS   = [
    'male', 'female', 'homosexual_gay_or_lesbian', 'christian', 'jewish',
    'muslim', 'black', 'white', 'psychiatric_or_mental_illness'
]
AUX_TOXICITY_COLUMNS = [
    'severe_toxicity', 'obscene', 'identity_attack',
    'insult', 'threat', 'sexual_explicit'
]

OUT_DIR = Path('/content/output')
OUT_DIR.mkdir(exist_ok=True)

def seed_everything(seed=1234):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(1234)


## 4. Evaluation Metric (JigsawEvaluator)

In [ ]:
class JigsawEvaluator:
    """
    Reproduces the competition metric:
      score = 0.25 * overall_AUC + 0.75 * mean(power_mean_p=-5 of [subgroup, BPSN, BNSP] AUCs)
    """

    def __init__(self, y_true: np.ndarray, y_identity: np.ndarray,
                 power: int = -5, overall_weight: float = 0.25):
        self.y   = (y_true >= 0.5).astype(int)
        self.y_i = (y_identity >= 0.5).astype(int)
        self.n_subgroups  = self.y_i.shape[1]
        self.power        = power
        self.overall_weight = overall_weight

    @staticmethod
    def _auc(y_true, y_pred):
        try:
            return roc_auc_score(y_true, y_pred)
        except ValueError:
            return 1e-15

    def _subgroup_auc(self, i, y_pred):
        mask = self.y_i[:, i] == 1
        return self._auc(self.y[mask], y_pred[mask])

    def _bpsn_auc(self, i, y_pred):
        # Background-Positive Subgroup-Negative
        # toxic outside subgroup  vs  non-toxic inside subgroup
        mask = self.y_i[:, i] + self.y == 1
        return self._auc(self.y[mask], y_pred[mask])

    def _bnsp_auc(self, i, y_pred):
        # Background-Negative Subgroup-Positive
        # non-toxic outside subgroup  vs  toxic inside subgroup
        mask = self.y_i[:, i] + self.y != 1
        return self._auc(self.y[mask], y_pred[mask])

    def _power_mean(self, arr):
        return np.power(np.mean(np.power(arr, self.power)), 1 / self.power)

    def get_score(self, y_pred: np.ndarray):
        sub  = np.zeros(self.n_subgroups)
        bpsn = np.zeros(self.n_subgroups)
        bnsp = np.zeros(self.n_subgroups)
        for i in range(self.n_subgroups):
            sub[i]  = self._subgroup_auc(i, y_pred)
            bpsn[i] = self._bpsn_auc(i, y_pred)
            bnsp[i] = self._bnsp_auc(i, y_pred)

        bias_score   = np.mean([self._power_mean(sub),
                                self._power_mean(bpsn),
                                self._power_mean(bnsp)])
        overall_auc  = roc_auc_score(self.y, y_pred)
        final        = self.overall_weight * overall_auc + (1 - self.overall_weight) * bias_score

        return final, {
            'overall_auc': overall_auc,
            'mean_subgroup_auc': self._power_mean(sub),
            'mean_bpsn_auc':     self._power_mean(bpsn),
            'mean_bnsp_auc':     self._power_mean(bnsp),
            'final_score':       final,
            'subgroup_auc':  dict(zip(IDENTITY_COLUMNS, sub)),
            'bpsn_auc':      dict(zip(IDENTITY_COLUMNS, bpsn)),
            'bnsp_auc':      dict(zip(IDENTITY_COLUMNS, bnsp)),
        }


## 5. Sample Weighting

In [ ]:
def training_weights(df, toxicity_col, identity_cols):
    """
    Weight = number of AUC sub-scores each sample contributes to (normalised to [0.25, 1.0]).
    Directly aligns training signal with the evaluation metric structure.

    Components:
      +1/4  always (overall AUC)
      +1/4  if example mentions any identity     -> subgroup AUC
      +1/4  if toxic background + non-toxic subgroup  -> BPSN AUC
      +1/4  if non-toxic background + toxic subgroup  -> BNSP AUC
    """
    id_vals = df[identity_cols].fillna(0).values
    tox     = df[toxicity_col].values

    subgroup_pos = (id_vals >= 0.5).any(axis=1).astype(int)   # mentions an identity & is toxic
    subgroup_neg = (id_vals <  0.5).all(axis=1).astype(int)   # non-toxic identity mention

    bg_pos = (tox >= 0.5).astype(int)
    bg_neg = (tox <  0.5).astype(int)

    w = np.ones(len(df)) / 4                            # overall AUC component
    w += (id_vals >= 0.5).mean(axis=1) / 4              # subgroup AUC component
    w += ((bg_pos + subgroup_neg) > 1).astype(float) / 4  # BPSN component
    w += ((bg_neg + subgroup_pos) > 1).astype(float) / 4  # BNSP component
    return w


## 6. Custom Loss Function

In [ ]:
class CustomLoss(nn.Module):
    """
    Multi-task BCE loss with:
      - Per-sample weighting (from training_weights)
      - Optional Beta-prior label smoothing based on annotator count:
            y_smooth = (y * n + alpha) / (n + alpha + beta)
        This shrinks confident labels for low-annotator-count examples.
      - Optional log(n+2) instance weighting by annotator count.
    """

    def __init__(self, loss_weight=None, alpha=1.0, beta=1.0,
                 use_annotator_smoothing=False, annotator_count_weighting=False):
        super().__init__()
        self.loss_weight             = loss_weight
        self.alpha                   = alpha
        self.beta                    = beta
        self.use_annotator_smoothing = use_annotator_smoothing
        self.annotator_count_weighting = annotator_count_weighting

    def forward(self, logits, targets, annotator_counts=None):
        """
        logits   : (B, 1 + n_aux)
        targets  : (B, 2 + n_aux)  — col0=target, col1=sample_weight, col2+=aux_targets
        annotator_counts : (B,) or None
        """
        sample_w   = targets[:, 1:2] if self.loss_weight is not None else None
        lw         = self.loss_weight if self.loss_weight is not None else 1.0

        if not self.use_annotator_smoothing or annotator_counts is None:
            primary_tgt = targets[:, :1]
            aux_tgt     = targets[:, 2:]
        else:
            n = annotator_counts.view(-1, 1)
            primary_tgt = (targets[:, :1] * n + self.alpha) / (n + self.alpha + self.beta)
            n_aux = targets[:, 2:].size(1)
            n_rep = n.repeat(1, n_aux)
            aux_tgt = (targets[:, 2:] * n_rep + self.alpha) / (n_rep + self.alpha + self.beta)

        loss1 = nn.BCEWithLogitsLoss(weight=sample_w, reduction='none')(logits[:, :1], primary_tgt)
        loss2 = nn.BCEWithLogitsLoss(reduction='none')(logits[:, 1:], aux_tgt).mean(dim=1, keepdim=True)
        combined = loss1 * lw + loss2

        if self.annotator_count_weighting and annotator_counts is not None:
            n = annotator_counts.view(-1, 1)
            aw = torch.log(n + 2)
            return (combined * aw).mean()
        return combined.mean()


## 7. Dataset & DataLoader

In [ ]:
class TextDataset(torch.utils.data.Dataset):

    def __init__(self, token_lists, targets=None, identities=None, annotator_counts=None):
        self.token_lists      = token_lists
        self.targets          = targets
        self.identities       = identities
        self.annotator_counts = annotator_counts

    def __len__(self):
        return len(self.token_lists)

    def __getitem__(self, idx):
        if self.targets is None:
            return self.token_lists[idx], idx
        return (self.token_lists[idx], idx,
                self.annotator_counts[idx],
                self.targets[idx],
                self.identities[idx])

    def collate_fn(self, batch):
        transposed = list(zip(*batch))
        max_len = max(len(x) for x in transposed[0])
        tokens = np.zeros((len(batch), max_len), dtype=np.int64)
        for i, row in enumerate(transposed[0]):
            tokens[i, :len(row)] = np.array(row[:max_len])
        tensors = [
            torch.from_numpy(tokens),
            torch.tensor(transposed[1], dtype=torch.int32),
        ]
        for i in range(2, len(transposed)):
            tensors.append(torch.tensor(transposed[i], dtype=torch.float32))
        return tensors


class LengthBucketingDataLoader:
    """Groups sequences by length to minimise padding within batches."""

    def __init__(self, dataset, batch_size=16, shuffle=False, drop_last=False):
        self.dataset    = dataset
        self.batch_size = batch_size
        self.shuffle    = shuffle
        self.drop_last  = drop_last

    def __iter__(self):
        indices = list(range(len(self.dataset)))
        if self.shuffle:
            random.shuffle(indices)

        # bucket: sort within chunks of 100×batch_size
        bucket = self.batch_size * 100
        batches = []
        for start in range(0, len(indices), bucket):
            chunk = indices[start:start + bucket]
            chunk.sort(key=lambda i: len(self.dataset.token_lists[i]))
            for b_start in range(0, len(chunk), self.batch_size):
                b = chunk[b_start:b_start + self.batch_size]
                if len(b) == self.batch_size or not self.drop_last:
                    batches.append(b)

        if self.shuffle:
            random.shuffle(batches)

        for batch_idx in batches:
            items = [self.dataset[i] for i in batch_idx]
            yield self.dataset.collate_fn(items)

    def __len__(self):
        return len(self.dataset) // self.batch_size


## 8. BERT Fine-tuning
Uses `bert-large-uncased` with:
- Head+tail truncation (first 128 + last 90 tokens for max_len=220)
- 2 epochs with different negative-downsampling halves per epoch
- Multi-task output (1 primary + 6 auxiliary toxicity heads)
- Linear warmup + linear decay LR schedule
- Mixed precision via `torch.cuda.amp`


In [ ]:
from pytorch_pretrained_bert import BertTokenizer, BertForSequenceClassification

# ── Tokenizer with head+tail truncation ──────────────────────────────────────
class HeadTailTokenizer:
    def __init__(self, bert_tokenizer, max_len=220, max_head_len=128):
        self.tok         = bert_tokenizer
        self.max_len     = max_len - 2          # reserve [CLS] and [SEP]
        self.max_head_len = max_head_len

    def encode(self, text: str):
        tokens = self.tok.tokenize(str(text))
        if len(tokens) > self.max_len:
            head = tokens[:self.max_head_len]
            tail = tokens[self.max_head_len - self.max_len:]   # last N tokens
            tokens = head + tail
        tokens = ['[CLS]'] + tokens + ['[SEP]']
        return self.tok.convert_tokens_to_ids(tokens)

    def batch_encode(self, texts, num_workers=4):
        from multiprocessing.pool import Pool
        with Pool(num_workers) as pool:
            return list(tqdm(pool.imap(self.encode, texts, chunksize=2000),
                             total=len(texts), desc='Tokenising'))


In [ ]:
# ── Negative downsampling helper ─────────────────────────────────────────────
def get_negative_indices(y_train, y_identity):
    """Pure negatives: target=0 AND all identity columns=0."""
    target_neg   = (y_train[:, 0] < 0.5)
    identity_neg = (y_identity < 0.5).all(axis=1)
    return np.where(target_neg & identity_neg)[0]


In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
def train_bert(model, loss_fn, X_train, y_train, y_identity_train,
               y_annotator_counts_train, X_valid, y_valid, y_identity_valid,
               evaluator, config, device):

    from pytorch_pretrained_bert import BertAdam

    loss_weight = 1.0 / y_train[:, 1].mean()
    loss_fn_weighted = CustomLoss(loss_weight)

    negative_indices = get_negative_indices(y_train, y_identity_train)
    np.random.shuffle(negative_indices)
    frac = config.get('down_sample_frac', 0.5)
    split = int(len(negative_indices) * frac)
    drop_sets = [
        set(negative_indices[:split]),
        set(negative_indices[split:]),
    ]

    # Effective training size (epoch 0, worst case)
    len_train = len(y_train) - len(drop_sets[0])

    num_steps = int(config['epochs'] * len_train /
                    config['batch_size'] / config['accumulation_steps'])

    no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']
    opt_params = [
        {'params': [p for n, p in model.named_parameters()
                    if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
        {'params': [p for n, p in model.named_parameters()
                    if any(nd in n for nd in no_decay)],     'weight_decay': 0.0},
    ]
    optimizer = BertAdam(opt_params, lr=config['lr'],
                         warmup=config['warmup'], t_total=num_steps)

    scaler = torch.cuda.amp.GradScaler()
    best_score, best_state = 0.0, None

    for epoch, drop_idx in zip(range(config['epochs']), drop_sets):
        sample_idx = [i for i in range(len(y_train)) if i not in drop_idx]
        train_ds   = TextDataset(
            [X_train[i] for i in sample_idx],
            y_train[sample_idx],
            y_identity_train[sample_idx],
            y_annotator_counts_train[sample_idx],
        )
        loader = LengthBucketingDataLoader(
            train_ds, batch_size=config['batch_size'], shuffle=True, drop_last=True)

        model.train()
        optimizer.zero_grad()
        running_loss = 0.0

        for step, (x, _, a, y, _) in enumerate(tqdm(loader, desc=f'Epoch {epoch+1}')):
            x, y = x.to(device), y.to(device)
            with torch.cuda.amp.autocast():
                logits = model(x, attention_mask=(x > 0), labels=None)
                loss   = loss_fn_weighted(logits, y)
            scaler.scale(loss / config['accumulation_steps']).backward()
            running_loss += loss.item()
            if (step + 1) % config['accumulation_steps'] == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

        print(f'  Epoch {epoch+1} avg loss: {running_loss / (step+1):.4f}')

        # Validation
        val_score = evaluate_bert(model, X_valid, y_valid, y_identity_valid,
                                  evaluator, config['batch_size'], device)
        print(f'  Validation score: {val_score:.5f}')
        if val_score > best_score:
            best_score = val_score
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model, best_score


def evaluate_bert(model, X, y_raw, y_identity, evaluator, batch_size, device):
    preds = predict_bert(model, X, batch_size, device)
    score, _ = evaluator.get_score(preds)
    return score


def predict_bert(model, X, batch_size, device):
    model.eval()
    ds = TextDataset(X)
    loader = LengthBucketingDataLoader(ds, batch_size=batch_size, shuffle=False)
    all_preds, all_idx = [], []
    with torch.no_grad():
        for x, idx in loader:
            with torch.cuda.amp.autocast():
                logits = model(x.to(device), attention_mask=(x.to(device) > 0), labels=None)
            all_preds.append(torch.sigmoid(logits[:, 0]).cpu().numpy())
            all_idx.append(idx.numpy())
    preds = np.concatenate(all_preds)
    idx   = np.concatenate(all_idx)
    return preds[np.argsort(idx)]


In [ ]:
# ── Run BERT training ─────────────────────────────────────────────────────────
BERT_CONFIG = {
    'lm_model_name':    'bert-large-uncased',
    'max_len':          220,
    'max_head_len':     128,
    'epochs':           2,
    'down_sample_frac': 0.5,
    'lr':               1.5e-5,
    'batch_size':       16,
    'accumulation_steps': 4,
    'warmup':           0.05,
    'valid_size':       200_000,
    'seed':             1234,
}

seed_everything(BERT_CONFIG['seed'])

print('Loading data...')
df_all = pd.read_csv(TRAIN_PATH).sample(frac=1, random_state=BERT_CONFIG['seed']).reset_index(drop=True)
df_all['comment_text'] = df_all['comment_text'].astype(str).fillna('DUMMY')
df_all = df_all.fillna(0)

valid_size  = BERT_CONFIG['valid_size']
df_tr       = df_all.iloc[:-valid_size].reset_index(drop=True)
df_val      = df_all.iloc[-valid_size:].reset_index(drop=True)

print(f'Train: {len(df_tr):,}  |  Valid: {len(df_val):,}')

print('Tokenising...')
bert_tokenizer = BertTokenizer.from_pretrained(
    BERT_CONFIG['lm_model_name'], do_lower_case=True)
tokenizer = HeadTailTokenizer(bert_tokenizer,
                               max_len=BERT_CONFIG['max_len'],
                               max_head_len=BERT_CONFIG['max_head_len'])

X_tr  = tokenizer.batch_encode(df_tr['comment_text'].tolist())
X_val = tokenizer.batch_encode(df_val['comment_text'].tolist())

df_test = pd.read_csv(TEST_PATH)
df_test['comment_text'] = df_test['comment_text'].astype(str).fillna('DUMMY')
X_test = tokenizer.batch_encode(df_test['comment_text'].tolist())

weights_tr = training_weights(df_tr, TOXICITY_COLUMN, IDENTITY_COLUMNS)
y_tr = np.hstack([
    df_tr[TOXICITY_COLUMN].values.reshape(-1, 1),
    weights_tr.reshape(-1, 1),
    df_tr[AUX_TOXICITY_COLUMNS].values,
])
y_identity_tr         = df_tr[IDENTITY_COLUMNS].values
y_annotator_counts_tr = df_tr['toxicity_annotator_count'].values

y_val          = df_val[TOXICITY_COLUMN].values
y_identity_val = df_val[IDENTITY_COLUMNS].values

evaluator = JigsawEvaluator(y_val, y_identity_val)

print('Building model...')
bert_model = BertForSequenceClassification.from_pretrained(
    BERT_CONFIG['lm_model_name'],
    num_labels=1 + len(AUX_TOXICITY_COLUMNS)
).to(DEVICE)

loss_fn = CustomLoss()  # weights injected inside train_bert

bert_model, bert_best_score = train_bert(
    bert_model, loss_fn,
    X_tr, y_tr, y_identity_tr, y_annotator_counts_tr,
    X_val, y_val, y_identity_val,
    evaluator, BERT_CONFIG, DEVICE,
)
print(f'\nBest BERT validation score: {bert_best_score:.5f}')
torch.save(bert_model.state_dict(), OUT_DIR / 'bert_large_uncased.pt')


Loading data...
Train: 1,604,874  |  Valid: 200,000
Tokenising...


Tokenising:   0%|          | 0/1604874 [00:00<?, ?it/s]

Tokenising:   0%|          | 0/200000 [00:00<?, ?it/s]

Tokenising:   0%|          | 0/97320 [00:00<?, ?it/s]

Building model...


Epoch 1:   0%|          | 0/57958 [00:00<?, ?it/s]

  Epoch 1 avg loss: 0.4431
  Validation score: 0.94140


Epoch 2:   0%|          | 0/57958 [00:00<?, ?it/s]

  Epoch 2 avg loss: 0.4182
  Validation score: 0.94505

Best BERT validation score: 0.94505


In [ ]:
# Predict on test set
bert_test_preds = predict_bert(bert_model, X_test, BERT_CONFIG['batch_size'], DEVICE)
pd.DataFrame({'id': df_test['id'], 'prediction': bert_test_preds}).to_csv(
    OUT_DIR / 'bert_test_preds.csv', index=False)
print('BERT test predictions saved.')


BERT test predictions saved.


## 9. LSTM-GRU Baseline (LSTM-f)
Uses GloVe + FastText embeddings projected and summed into a shared embedding space,
followed by Bi-LSTM → Bi-GRU → global avg+max pool → dense head.
5-fold CV with Beta-smoothed annotator-count loss.


In [ ]:
import re, string

# ── Text preprocessing ────────────────────────────────────────────────────────
MISSPELLINGS = {
    "aren't":"are not","can't":"cannot","couldn't":"could not","didn't":"did not",
    "doesn't":"does not","don't":"do not","hadn't":"had not","hasn't":"has not",
    "haven't":"have not","he'd":"he would","he'll":"he will","he's":"he is",
    "i'd":"I had","i'll":"I will","i'm":"I am","isn't":"is not","it's":"it is",
    "it'll":"it will","i've":"I have","let's":"let us","mightn't":"might not",
    "mustn't":"must not","shan't":"shall not","she'd":"she would","she'll":"she will",
    "she's":"she is","shouldn't":"should not","that's":"that is","there's":"there is",
    "they'd":"they would","they'll":"they will","they're":"they are","they've":"they have",
    "we'd":"we would","we're":"we are","weren't":"were not","we've":"we have",
    "what'll":"what will","what're":"what are","what's":"what is","what've":"what have",
    "where's":"where is","who'd":"who would","who'll":"who will","who're":"who are",
    "who's":"who is","who've":"who have","won't":"will not","wouldn't":"would not",
    "you'd":"you would","you'll":"you will","you're":"you are","you've":"you have",
    "'re":" are","wasn't":"was not","we'll":" will","tryin'":"trying",
}
MISSPELL_RE = re.compile('(%s)' % '|'.join(MISSPELLINGS.keys()))
PUNCTS = list(''',.":)(-!?|;'$&/[]>%=#*+\\•~@£·_{}©^®`<→°€™›♥←×§″′Â█½à…"★"–●â►−¢²¬░¶↑±¿▾═¦║―¥▓—‹─▒：¼⊕▼▪†■'▀¨▄♫☆é¯♦¤▲è¸¾Ã⋅'∞∙）↓、│（»，♪╩╚³・╦╣╔╗▬❤ïØ¹≤‡√''')

def preprocess_text(text):
    text = str(text).lower()
    text = MISSPELL_RE.sub(lambda m: MISSPELLINGS[m.group(0)], text)
    for p in PUNCTS + list(string.punctuation):
        if p in text:
            text = text.replace(p, f' {p} ')
    text = re.sub(r'\d+', ' ', text)
    return text.strip()

# ── Vocab & tokenization ──────────────────────────────────────────────────────
def build_vocab(texts, max_features=100_000):
    counter = Counter()
    for t in texts:
        counter.update(t.split())
    token2id = {'<PAD>': 0, '<UNK>': max_features + 1}
    token2id.update({tok: i + 1 for i, (tok, _) in
                     enumerate(counter.most_common(max_features))})
    return token2id

def tokenize_texts(texts, token2id, max_len=220):
    unk = len(token2id) - 1
    return [[token2id.get(t, unk) for t in txt.split()[:max_len]] for txt in texts]


In [ ]:
# ── Load GloVe / FastText ─────────────────────────────────────────────────────
# Download embeddings if not already present
GLOVE_PATH    = Path('/content/glove.840B.300d.txt')
FASTTEXT_PATH = Path('/content/crawl-300d-2M.vec')

if not GLOVE_PATH.exists():
    print("Downloading GloVe 840B 300d...")
    !wget -q --show-progress "https://nlp.stanford.edu/data/glove.840B.300d.zip" -O /tmp/glove.zip
    !unzip -q /tmp/glove.zip -d /content/
    print("Done.")

if not FASTTEXT_PATH.exists():
    print("Downloading FastText crawl-300d-2M...")
    !wget -q --show-progress "https://dl.fbaipublicfiles.com/fasttext/vectors-english/crawl-300d-2M.vec.zip" -O /tmp/ft.zip
    !unzip -q /tmp/ft.zip -d /content/
    print("Done.")

def load_embedding_matrix(path, token2id, embed_size=300):
    from nltk.stem import PorterStemmer, SnowballStemmer
    from nltk.stem.lancaster import LancasterStemmer
    ps, lc, sb = PorterStemmer(), LancasterStemmer(), SnowballStemmer('english')

    index = {}
    with open(path, encoding='utf-8', errors='ignore') as f:
        for line in tqdm(f, desc=f'Loading {Path(path).name}'):
            parts = line.rstrip().split(' ')
            if len(parts) == embed_size + 1:
                index[parts[0]] = np.array(parts[1:], dtype=np.float32)

    nb_words = min(len(token2id), max(token2id.values()) + 1)
    matrix   = np.zeros((nb_words, embed_size), dtype=np.float32)
    for word, i in token2id.items():
        if i >= nb_words:
            continue
        for candidate in [word, word.lower(), word.upper(), word.capitalize(),
                          ps.stem(word), lc.stem(word), sb.stem(word)]:
            if candidate in index:
                matrix[i] = index[candidate]
                break
    return matrix


/tmp/glove.zip      100%[===================>]   2.03G  5.10MB/s    in 7m 3s   
Done.
/tmp/ft.zip         100%[===================>]   1.42G   210MB/s    in 6.8s    
Done.


In [ ]:
# ── Model architecture ────────────────────────────────────────────────────────
class SpatialDropout(nn.Dropout2d):
    def forward(self, x):
        x = x.unsqueeze(2).permute(0, 3, 2, 1)
        x = super().forward(x)
        return x.permute(0, 3, 2, 1).squeeze(2)

class ProjSumEmbedding(nn.Module):
    """Projects each pre-trained embedding to output_size and sums them."""
    def __init__(self, matrices, output_size):
        super().__init__()
        self.projectors = nn.ModuleList()
        for m in matrices:
            emb = nn.Embedding(*m.shape)
            emb.weight = nn.Parameter(torch.tensor(m, dtype=torch.float32))
            emb.weight.requires_grad = False
            proj = nn.Linear(m.shape[1], output_size)
            nn.init.xavier_uniform_(proj.weight)
            self.projectors.append(nn.Sequential(emb, proj))

    def forward(self, x):
        return F.relu(sum(p(x) for p in self.projectors))


class LstmGruNet(nn.Module):
    """BiLSTM → BiGRU → avg+max pool → dense → multi-task output."""
    def __init__(self, embedding_matrices, num_aux_targets,
                 embed_proj_size=256, lstm_units=128, gru_units=128):
        super().__init__()
        self.embedding         = ProjSumEmbedding(embedding_matrices, embed_proj_size)
        self.embedding_dropout = SpatialDropout(0.2)
        self.lstm = nn.LSTM(embed_proj_size, lstm_units, bidirectional=True, batch_first=True)
        self.gru  = nn.GRU(lstm_units * 2, gru_units, bidirectional=True, batch_first=True)
        dense_in  = gru_units * 4
        self.fc1  = nn.Linear(dense_in, dense_in)
        self.fc2  = nn.Linear(dense_in, dense_in)
        self.out_main = nn.Linear(dense_in, 1)
        self.out_aux  = nn.Linear(dense_in, num_aux_targets)

    def forward(self, x):
        h = self.embedding_dropout(self.embedding(x))
        h, _ = self.lstm(h)
        h, _ = self.gru(h)
        avg_pool = h.mean(dim=1)
        max_pool = h.max(dim=1).values
        h = torch.cat([avg_pool, max_pool], dim=1)
        h = h + F.relu(self.fc1(h)) + F.relu(self.fc2(h))
        return torch.cat([self.out_main(h), self.out_aux(h)], dim=1)


In [ ]:
# ── LSTM training loop ────────────────────────────────────────────────────────
def train_lstm_fold(model, loss_fn, train_ds, valid_ds, device,
                    batch_size=512, lr=1e-3, max_epochs=10, patience=10):

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=2.5e-5)
    best_score, best_state, no_improve = 0.0, None, 0

    for epoch in range(max_epochs):
        model.train()
        for x, _, a, y, y_id in LengthBucketingDataLoader(
                train_ds, batch_size=batch_size, shuffle=True, drop_last=True):
            x, a, y = x.long().to(device), a.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss   = loss_fn(logits, y, a)
            loss.backward()
            optimizer.step()

        # Validate
        model.eval()
        val_preds, val_ys, val_yids = [], [], []
        with torch.no_grad():
            for x, _, a, y, y_id in LengthBucketingDataLoader(
                    valid_ds, batch_size=batch_size, shuffle=False):
                logits = model(x.long().to(device))
                val_preds.append(torch.sigmoid(logits[:, 0]).cpu().numpy())
                val_ys.append(y[:, 0].numpy())
                val_yids.append(y_id.numpy())

        val_preds = np.concatenate(val_preds)
        val_ys    = np.concatenate(val_ys)
        val_yids  = np.concatenate(val_yids).reshape(-1, len(IDENTITY_COLUMNS))
        ev        = JigsawEvaluator(val_ys, val_yids)
        score, _  = ev.get_score(val_preds)
        print(f'  Epoch {epoch+1}: val_score={score:.5f}')

        if score > best_score:
            best_score = score
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print('  Early stopping.')
                break

    model.load_state_dict(best_state)
    return model, best_score


In [ ]:
# ── Run LSTM training ─────────────────────────────────────────────────────────
LSTM_CONFIG = {
    'max_len':      220,
    'max_features': 100_000,
    'batch_size':   512,
    'lr':           1e-3,
    'max_epochs':   10,
    'patience':     5,
    'num_folds':    5,
    'loss_alpha':   0.1,
    'loss_beta':    1.0,
    'seed':         1029,
}

seed_everything(LSTM_CONFIG['seed'])

print('Preprocessing text...')
df_tr_lstm = pd.read_csv(TRAIN_PATH).sample(frac=1, random_state=LSTM_CONFIG['seed']).reset_index(drop=True)
df_tr_lstm = df_tr_lstm.fillna(0)
valid_size = 200_000
df_val_lstm = df_tr_lstm.tail(valid_size).reset_index(drop=True)
df_tr_lstm  = df_tr_lstm.head(len(df_tr_lstm) - valid_size).reset_index(drop=True)

texts_tr  = df_tr_lstm['comment_text'].astype(str).apply(preprocess_text).tolist()
texts_val = df_val_lstm['comment_text'].astype(str).apply(preprocess_text).tolist()

df_test_lstm = pd.read_csv(TEST_PATH).fillna(0)
texts_test = df_test_lstm['comment_text'].astype(str).apply(preprocess_text).tolist()

token2id = build_vocab(texts_tr, LSTM_CONFIG['max_features'])
X_tr_lstm   = tokenize_texts(texts_tr,  token2id, LSTM_CONFIG['max_len'])
X_val_lstm  = tokenize_texts(texts_val, token2id, LSTM_CONFIG['max_len'])
X_test_lstm = tokenize_texts(texts_test, token2id, LSTM_CONFIG['max_len'])

print('Loading embeddings...')
glove_matrix    = load_embedding_matrix(GLOVE_PATH,    token2id)
fasttext_matrix = load_embedding_matrix(FASTTEXT_PATH, token2id)

weights_tr_lstm = training_weights(df_tr_lstm, TOXICITY_COLUMN, IDENTITY_COLUMNS)
loss_weight_lstm = 1.0 / weights_tr_lstm.mean()

y_tr_lstm = np.hstack([
    df_tr_lstm[TOXICITY_COLUMN].values.reshape(-1, 1),
    weights_tr_lstm.reshape(-1, 1),
    df_tr_lstm[AUX_TOXICITY_COLUMNS].values,
])
y_identity_tr_lstm         = df_tr_lstm[IDENTITY_COLUMNS].values
y_annotator_counts_tr_lstm = df_tr_lstm['toxicity_annotator_count'].values

y_val_lstm         = df_val_lstm[TOXICITY_COLUMN].values
y_identity_val_lstm = df_val_lstm[IDENTITY_COLUMNS].values

lstm_loss_fn = CustomLoss(
    loss_weight=loss_weight_lstm,
    alpha=LSTM_CONFIG['loss_alpha'],
    beta=LSTM_CONFIG['loss_beta'],
    use_annotator_smoothing=True,
    annotator_count_weighting=True,
)

all_related = [TOXICITY_COLUMN] + AUX_TOXICITY_COLUMNS + IDENTITY_COLUMNS
neg_idx_lstm = np.where(
    (df_tr_lstm[all_related] == 0.0).sum(axis=1) == len(all_related)
)[0]

skf = StratifiedKFold(n_splits=LSTM_CONFIG['num_folds'], shuffle=True, random_state=1)
y_bin = (y_tr_lstm[:, 0] >= 0.5).astype(int)

lstm_test_preds = np.zeros(len(X_test_lstm))
pred_count = 0

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tr_lstm, y_bin)):
    print(f'\n=== LSTM Fold {fold+1}/{LSTM_CONFIG["num_folds"]} ===')
    seed_everything(fold)

    # Negative downsampling
    np.random.shuffle(neg_idx_lstm)
    drop = set(neg_idx_lstm[:len(neg_idx_lstm) // 2])
    tr_idx_sampled = [i for i in tr_idx if i not in drop]

    train_ds = TextDataset(
        [X_tr_lstm[i] for i in tr_idx_sampled],
        y_tr_lstm[tr_idx_sampled],
        y_identity_tr_lstm[tr_idx_sampled],
        y_annotator_counts_tr_lstm[tr_idx_sampled],
    )
    valid_ds = TextDataset(
        [X_tr_lstm[i] for i in val_idx],
        y_tr_lstm[val_idx],
        y_identity_tr_lstm[val_idx],
        y_annotator_counts_tr_lstm[val_idx],
    )

    lstm_model = LstmGruNet(
        embedding_matrices=[glove_matrix, fasttext_matrix],
        num_aux_targets=len(AUX_TOXICITY_COLUMNS),
    ).to(DEVICE)

    lstm_model, fold_score = train_lstm_fold(
        lstm_model, lstm_loss_fn, train_ds, valid_ds, DEVICE,
        batch_size=LSTM_CONFIG['batch_size'],
        lr=LSTM_CONFIG['lr'],
        max_epochs=LSTM_CONFIG['max_epochs'],
        patience=LSTM_CONFIG['patience'],
    )
    print(f'  Fold {fold+1} best score: {fold_score:.5f}')

    # Test inference
    test_ds = TextDataset(X_test_lstm)
    test_preds_fold = []
    test_idx_fold   = []
    lstm_model.eval()
    with torch.no_grad():
        for x, idx in LengthBucketingDataLoader(test_ds, batch_size=512, shuffle=False):
            logits = lstm_model(x.long().to(DEVICE))
            test_preds_fold.append(torch.sigmoid(logits[:, 0]).cpu().numpy())
            test_idx_fold.append(idx.numpy())
    test_preds_fold = np.concatenate(test_preds_fold)
    test_idx_fold   = np.concatenate(test_idx_fold)
    test_preds_fold = test_preds_fold[np.argsort(test_idx_fold)]

    lstm_test_preds += test_preds_fold
    pred_count += 1

    torch.save(lstm_model.state_dict(), OUT_DIR / f'lstm_fold{fold}.pt')

lstm_test_preds /= pred_count
pd.DataFrame({'id': df_test_lstm['id'], 'prediction': lstm_test_preds}).to_csv(
    OUT_DIR / 'lstm_test_preds.csv', index=False)
print('\nLSTM test predictions saved.')


Preprocessing text...
Loading embeddings...


Loading glove.840B.300d.txt: 0it [00:00, ?it/s]

Loading crawl-300d-2M.vec: 0it [00:00, ?it/s]


=== LSTM Fold 1/5 ===
  Epoch 1: val_score=0.92588
  Epoch 2: val_score=0.92987
  Epoch 3: val_score=0.93114
  Epoch 4: val_score=0.93205
  Epoch 5: val_score=0.93324
  Epoch 6: val_score=0.93363
  Epoch 7: val_score=0.93314
  Epoch 8: val_score=0.93492
  Epoch 9: val_score=0.93514
  Epoch 10: val_score=0.93487
  Fold 1 best score: 0.93514

=== LSTM Fold 2/5 ===
  Epoch 1: val_score=0.92246
  Epoch 2: val_score=0.92723
  Epoch 3: val_score=0.92975
  Epoch 4: val_score=0.92931
  Epoch 5: val_score=0.93034
  Epoch 6: val_score=0.93008
  Epoch 7: val_score=0.93172
  Epoch 8: val_score=0.93120
  Epoch 9: val_score=0.93196
  Epoch 10: val_score=0.93173
  Fold 2 best score: 0.93196

=== LSTM Fold 3/5 ===
  Epoch 1: val_score=0.92431
  Epoch 2: val_score=0.92840
  Epoch 3: val_score=0.92957
  Epoch 4: val_score=0.93063
  Epoch 5: val_score=0.93148
  Epoch 6: val_score=0.93175
  Epoch 7: val_score=0.93288
  Epoch 8: val_score=0.93228
  Epoch 9: val_score=0.93289
  Epoch 10: val_score=0.93373


## 10. Optuna Ensemble Blending
Uses Optuna to find optimal per-model weights on held-out validation data.
Runs 10 random splits, keeps only "robust" folds where train/valid gap < 0.03.


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def optuna_blend(model_preds_dict, y_true, y_identity,
                 n_folds=10, n_trials=300, threshold=0.03):
    """
    model_preds_dict : {model_name: np.ndarray of predictions}
    Returns mean weights over robust folds.
    """
    names = list(model_preds_dict.keys())
    preds_matrix = np.stack([model_preds_dict[n] for n in names], axis=1)  # (N, M)
    N = len(y_true)

    train_scores, valid_scores, fold_weights = [], [], []

    for fold in range(n_folds):
        rng = np.random.RandomState(fold)
        idx = rng.permutation(N)
        half = N // 2
        tr_idx, va_idx = idx[:half], idx[half:]

        tr_y    = y_true[tr_idx];      va_y    = y_true[va_idx]
        tr_yid  = y_identity[tr_idx];  va_yid  = y_identity[va_idx]
        tr_pred = preds_matrix[tr_idx]; va_pred = preds_matrix[va_idx]

        tr_ev = JigsawEvaluator(tr_y, tr_yid)
        va_ev = JigsawEvaluator(va_y, va_yid)

        def objective(trial):
            w = np.array([trial.suggest_float(n, 0.0, 1.0) for n in names])
            w /= w.sum() + 1e-12
            blended, _ = tr_ev.get_score(tr_pred @ w)
            return 1.0 - blended

        study = optuna.create_study(direction='minimize',
                                    sampler=optuna.samplers.TPESampler(seed=fold))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

        best_w = np.array([study.best_params[n] for n in names])
        best_w /= best_w.sum()

        tr_score, _ = tr_ev.get_score(tr_pred @ best_w)
        va_score, _ = va_ev.get_score(va_pred @ best_w)
        train_scores.append(tr_score)
        valid_scores.append(va_score)
        fold_weights.append(best_w)
        print(f'  Fold {fold+1}: train={tr_score:.5f}  valid={va_score:.5f}')

    robust = [i for i,(tr,va) in enumerate(zip(train_scores,valid_scores))
              if abs(tr - va) < threshold]
    print(f'\nRobust folds: {robust}')
    if not robust:
        print('No robust folds — using all.')
        robust = list(range(n_folds))

    final_weights = np.mean([fold_weights[i] for i in robust], axis=0)
    for n, w in zip(names, final_weights):
        print(f'  {n:<30s} {w:.4f}')
    return dict(zip(names, final_weights))


In [ ]:
# ── Load validation predictions for blending ──────────────────────────────────
# BERT valid preds (from training phase — re-predict if not cached)
bert_val_preds = predict_bert(bert_model, X_val, BERT_CONFIG['batch_size'], DEVICE)

# LSTM val preds — average across folds on the same held-out val split
# (Simplified: use the 200k validation split used during LSTM training)
# For a clean blend we need predictions on a common validation set.
# Here we run LSTM inference on df_val_lstm (same 200k block).

lstm_val_preds_list = []
for fold in range(LSTM_CONFIG['num_folds']):
    ckpt = OUT_DIR / f'lstm_fold{fold}.pt'
    if not ckpt.exists():
        continue
    m = LstmGruNet(
        embedding_matrices=[glove_matrix, fasttext_matrix],
        num_aux_targets=len(AUX_TOXICITY_COLUMNS),
    ).to(DEVICE)
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    m.eval()
    val_ds = TextDataset(X_val_lstm)
    preds, idxs = [], []
    with torch.no_grad():
        for x, idx in LengthBucketingDataLoader(val_ds, batch_size=512, shuffle=False):
            logits = m(x.long().to(DEVICE))
            preds.append(torch.sigmoid(logits[:, 0]).cpu().numpy())
            idxs.append(idx.numpy())
    p = np.concatenate(preds)[np.argsort(np.concatenate(idxs))]
    lstm_val_preds_list.append(p)

lstm_val_preds = np.mean(lstm_val_preds_list, axis=0)

# Use the common validation target (y_val from BERT split = df_all[-200k:])
# and y_val_lstm (same data, same rows)
blend_model_preds = {
    'bert_large_uncased': bert_val_preds,
    'lstm_gru':           lstm_val_preds,
}

print('Running Optuna blending...')
blend_weights = optuna_blend(
    blend_model_preds,
    y_val,          # from BERT split (same 200k block)
    y_identity_val,
    n_folds=10,
    n_trials=200,
    threshold=0.03,
)


Running Optuna blending...
  Fold 1: train=0.94699  valid=0.94291
  Fold 2: train=0.94434  valid=0.94555
  Fold 3: train=0.94708  valid=0.94302
  Fold 4: train=0.94680  valid=0.94304
  Fold 5: train=0.94320  valid=0.94687
  Fold 6: train=0.94547  valid=0.94465
  Fold 7: train=0.94497  valid=0.94510
  Fold 8: train=0.94520  valid=0.94464
  Fold 9: train=0.94481  valid=0.94526
  Fold 10: train=0.94261  valid=0.94738

Robust folds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  bert_large_uncased             0.9932
  lstm_gru                       0.0068


## 11. Final Submission

In [ ]:
# Blend test predictions with learned weights
bert_w = blend_weights.get('bert_large_uncased', 0.5)
lstm_w = blend_weights.get('lstm_gru', 0.5)

final_preds = bert_w * bert_test_preds + lstm_w * lstm_test_preds

submission = pd.DataFrame({
    'id':         df_test['id'],
    'prediction': final_preds,
})
submission.to_csv(OUT_DIR / 'submission.csv', index=False)
print(f'Submission saved: {len(submission):,} rows')
print(submission.head())


Submission saved: 97,320 rows
        id  prediction
0  7097320    0.011580
1  7097321    0.069213
2  7097322    0.332495
3  7097323    0.055701
4  7097324    0.032319


## 12. Bias Diagnostics

In [ ]:
# Quick sanity check: toxicity scores for identity-mention test sentences
test_sentences = [
    "I am a woman.",
    "I am a black woman.",
    "I am a gay man.",
    "I am a Muslim.",
    "I hate all Muslims.",
    "I am a white man.",
    "I am a Christian.",
    "People with mental illness are dangerous.",
    "She is Jewish and a wonderful person.",
    "You are being transphobic.",
    "I like prog metal",
]

bert_model.eval()
tok_out = [tokenizer.encode(s) for s in test_sentences]
ds = TextDataset(tok_out)
diag_preds = predict_bert(bert_model, tok_out, batch_size=8, device=DEVICE)

print(f'{"Sentence":<55} Toxicity%')
print('-' * 65)
for s, p in zip(test_sentences, diag_preds):
    print(f'{s:<55} {p*100:6.1f}%')


Sentence                                                Toxicity%
-----------------------------------------------------------------
I am a woman.                                              0.4%
I am a black woman.                                       10.0%
I am a gay man.                                            8.6%
I am a Muslim.                                             2.0%
I hate all Muslims.                                       87.6%
I am a white man.                                          3.2%
I am a Christian.                                          0.9%
People with mental illness are dangerous.                 38.8%
She is Jewish and a wonderful person.                      1.1%
You are being transphobic.                                41.9%
I like prog metal                                          0.7%
I play overwatch.                                          0.3%
I am a man                                                 0.3%
I am a gay man                      

In [ ]:
# Save generated outputs to Google Drive for persistence
print('Saving all generated outputs to Google Drive...')
DRIVE_OUT_DIR = DRIVE_DATA_DIR / 'output'
DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)

!cp "{OUT_DIR}/bert_test_preds.csv" "{DRIVE_OUT_DIR}/"
!cp "{OUT_DIR}/lstm_test_preds.csv" "{DRIVE_OUT_DIR}/"
!cp "{OUT_DIR}/submission.csv" "{DRIVE_OUT_DIR}/"
!cp "{OUT_DIR}/bert_large_uncased.pt" "{DRIVE_OUT_DIR}/"

# Copy LSTM fold checkpoints
for i in range(LSTM_CONFIG['num_folds']):
    lstm_ckpt_file = f'lstm_fold{i}.pt'
    !cp "{OUT_DIR}/{lstm_ckpt_file}" "{DRIVE_OUT_DIR}/"

print(f'Outputs saved to {DRIVE_OUT_DIR}')


Saving all generated outputs to Google Drive...
Outputs saved to /content/drive/MyDrive/jigsaw-data/output


## Faithfulness Analysis vs. Original 3rd Place Solution

Compared against [sakami0000/kaggle_jigsaw](https://github.com/sakami0000/kaggle_jigsaw) — the official 3rd place repo by team F.H.S.D.Y.

### Component Match

| Component | Original Repo | This Notebook | Match |
|---|---|---|---|
| BERT model | `bert-large-uncased` | `bert-large-uncased` | ✅ |
| max_len / max_head_len | 220 / 128 | 220 / 128 | ✅ |
| Head+tail truncation | first 128 + last 90 tokens | first 128 + last 90 tokens | ✅ |
| BERT epochs | 2 (must be ≤2 per README) | 2 | ✅ |
| Negative downsampling | 0.5 frac, different halves per epoch | 0.5 frac, alternating drop sets | ✅ |
| BERT lr / batch / accum / warmup | 1.5e-5 / 16 / 4 / 0.05 | 1.5e-5 / 16 / 4 / 0.05 | ✅ |
| Multi-task loss | primary + 6 aux toxicity heads | primary + 6 aux heads | ✅ |
| Sample weighting | Metric-aligned (subgroup/BPSN/BNSP) | Same 4-component formula | ✅ |
| LSTM-f architecture | ProjSum (GloVe+FastText) → BiLSTM → BiGRU → pool → dense | Identical | ✅ |
| LSTM-f hyperparams | max_len=220, features=100K, batch=512, lr=1e-3, 10 epochs, 5-fold | All match (patience 5 vs original 10) | ⚠️ |
| Beta-smoothed annotator loss | alpha/beta smoothing + log(n+2) weighting | alpha=0.1, beta=1.0, same weighting | ✅ |
| Optuna blending | n_folds=10, n_trials=300, threshold=0.03 | n_folds=10, n_trials=200, threshold=0.03 | ⚠️ |
| Evaluation metric | Custom Jigsaw evaluator (overall AUC + power-mean bias AUCs) | Faithfully reproduced | ✅ |
| BertAdam + linear warmup/decay | pytorch-pretrained-bert | Same | ✅ |
| Mixed precision | NVIDIA apex | torch.cuda.amp (modern equivalent) | ✅ |
| Sequence bucketing | Length-sorted batching | LengthBucketingDataLoader | ✅ |

### Not Implemented

| Component | Impact |
|---|---|
| GPT-2 fine-tuned variants | Original ensembled GPT-2 alongside BERT; adds model diversity |
| LSTM-s (second LSTM variant) | Cosine LR, pseudo-labeling, EMA, Capsule-Attention & Conv architectures |
| Old toxic data pre-fine-tuning | Optional BERT pre-fine-tune on prior Jigsaw competition data |
| Multiple BERT configs | Original ran bert-large-cased, bert-base-cased, bert-base-uncased separately |

This notebook recreates the **framing and core techniques** of the 3rd place solution, not the full ensemble. If required, the remaining model diversity will be added in the final audit project submission.

### Results

| Metric | Original (Private LB) | This Notebook (Local Val) |
|---|---|---|
| BERT alone | — | 0.94505 |
| LSTM-f best fold | — | 0.93514 |
| Blended score | **0.94683** | **~0.9474** |
| Blend weights | Multi-model ensemble | 99.3% BERT / 0.7% LSTM |

In [ ]:
# ── Compute Private Leaderboard Score ─────────────────────────────────────────
df_priv = pd.read_csv(DATA_DIR / 'test_private_expanded.csv')
df_priv = df_priv.fillna(0)

# Merge predictions with private test labels
preds_df = pd.read_csv(OUT_DIR / 'submission.csv')
df_eval = df_priv.merge(preds_df, on='id', how='inner')

print(f'Private test samples from the expanded test set on leaderboard: {len(df_eval):,}')

y_true_priv    = df_eval['toxicity'].values
y_identity_priv = df_eval[IDENTITY_COLUMNS].values
y_pred_priv    = df_eval['prediction'].values

evaluator_priv = JigsawEvaluator(y_true_priv, y_identity_priv)
priv_score, priv_details = evaluator_priv.get_score(y_pred_priv)

print(f'\nPrivate LB Score: {priv_score:.5f}')
print(f'  Overall AUC:        {priv_details["overall_auc"]:.5f}')
print(f'  Mean Subgroup AUC:  {priv_details["mean_subgroup_auc"]:.5f}')
print(f'  Mean BPSN AUC:      {priv_details["mean_bpsn_auc"]:.5f}')
print(f'  Mean BNSP AUC:      {priv_details["mean_bnsp_auc"]:.5f}')

print(f'\nOriginal 3rd place:   0.94683')
print(f'This notebook:        {priv_score:.5f}')
print(f'Difference:          {priv_score - 0.94683:+.5f}')

UsageError: Line magic function `%markdown` not found (But cell magic `%%markdown` exists, did you mean that instead?).


## 13. Private Leaderboard Evaluation

Our current solution's weakest component is Mean Subgroup AUC — the metric measuring performance within identity-mentioning comments. This is the most fairness-relevant metric in the competition scoring formula and the primary focus of our audit.